## imports and loading the data


In [1]:
from src.data_loader import load_data
import src.strategy as strategy
from src.backtester import VIXBacktester as bt

vega_Data = load_data('data/Vega_Data.xlsx')

vega_Data = vega_Data.set_index('Date')
vega_Data = vega_Data.loc['2018-01-01':'2026-03-31']   #slicing to cut out faulty data. vix values stay at 25.25 after thius date


## Calculations

In [2]:
# Execute the refactored modular strategy pipeline
vega_Data_old = vega_Data.copy()
vega_Data = strategy.run_strategy_pipeline(vega_Data, base_exposure=0.5)

display( vega_Data.head(30)  )

,ID,SVXY Open,SVXY Close,VXX Open,VXX Close,VIX,SPX,ALSI,ZAR,SVXY Returns,VXX Returns,Return Ratio (SVXY/VXX),Annualized Realized Volatility,Signal,VRP ( Realized - VIX ),Vol Ratio,Signal Strength,Base Exposure,Target Leverage
Date,,,,,,,,,,,,,,,,,,,
2018-01-01,12018,256.4199,256.4199,1852.000,1852.000,11.04,5212.76,8450.29,12.37240,NaN,NaN,NaN,NaN,Short,NaN,0.500000,1.000000,1.000000,1.000000
2018-01-02,12018,258.3799,265.1599,1842.399,1786.400,9.77,5256.28,8482.45,12.45830,0.034085,-0.035421,0.962270,NaN,Short,NaN,0.500000,1.000000,1.000000,1.000000
2018-01-03,12018,268.3799,271.0198,1765.600,1751.200,9.15,5289.92,8469.51,12.36395,0.022099,-0.019704,1.121549,NaN,Short,NaN,0.500000,1.000000,1.000000,1.000000
2018-01-04,12018,273.9199,271.7798,1732.800,1745.600,9.22,5312.33,8447.79,12.31270,0.002804,-0.003198,0.876921,NaN,Short,NaN,0.500000,1.000000,1.000000,1.000000
2018-01-05,12018,272.0999,271.8599,1741.600,1746.400,9.22,5349.69,8481.94,12.30440,0.000295,0.000458,-0.643088,NaN,Short,NaN,0.500000,1.000000,1.000000,1.000000
2018-01-08,12018,272.5999,275.0999,1741.600,1724.000,9.52,5358.68,8527.56,12.38500,0.011918,-0.012826,0.929171,NaN,Short,NaN,0.500000,1.000000,1.000000,1.000000
2018-01-09,12018,276.6599,272.1399,1712.800,1741.600,10.08,5367.26,8538.25,12.34500,-0.010760,0.010209,1.053964,NaN,Short,NaN,0.500000,1.000000,1.000000,1.000000
2018-01-10,12018,269.2197,274.8599,1762.400,1724.799,9.82,5361.33,8522.80,12.43460,0.009995,-0.009647,1.036072,NaN,Short,NaN,0.500000,1.000000,1.000000,1.000000
2018-01-11,12018,277.4600,276.4199,1708.800,1714.399,9.88,5399.46,8469.71,12.38400,0.005676,-0.006030,0.941279,NaN,Short,NaN,0.500000,1.000000,1.000000,1.000000


## Back testing

In [3]:
backtester = bt(vega_Data)

# Run and display the resulting DataFrame (simulation returns a copy)
vega_Data = backtester.run_simulation(vega_Data)


## Plotting values

In [4]:
import plotly.graph_objects as go

# Create the interactive figure
fig = go.Figure()

# Add Portfolio Value trace
fig.add_trace(go.Scatter(
    x=vega_Data.index,
    y=vega_Data['Portfolio Value'],
    name='Total Portfolio Value',
    line=dict(color='#0f978e', width=2.5),
    hovertemplate='Total Value: $%{y:,.2f}<extra></extra>'
))

# Add Vega Returns trace
fig.add_trace(go.Scatter(
    x=vega_Data.index,
    y=vega_Data['Position Value'],
    name='Position Value',
    line=dict(color='#e74c3c', width=2.0, dash='dash'),
    hovertemplate='Position Value: $%{y:,.2f}<extra></extra>'
))

# Add Cash trace
fig.add_trace(go.Scatter(
    x=vega_Data.index,
    y=vega_Data['Cash Balance'],
    name='Cash Balance',
    line=dict(color='#f39c12', width=2.0, dash='dot'),
    hovertemplate='Cash Balance: $%{y:,.2f}<extra></extra>'
))

# Add S&P 500 Buy & Hold benchmark trace
fig.add_trace(go.Scatter(
    x=vega_Data.index,
    y=vega_Data['SPX Buy Hold'],
    name='S&P 500 Buy & Hold',
    line=dict(color='#7f8c8d', width=2.0),
    hovertemplate='S&P 500 B&H: $%{y:,.2f}<extra></extra>'
))

# Update layout for modern aesthetics and interactivity
fig.update_layout(
    title=dict(
        text='VIX Trading Strategy vs S&P 500 Buy & Hold',
        font=dict(size=18, family='Arial', color='#2c3e50'),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title='Date',
        gridcolor='#eef2f3',
        showline=True,
        linecolor='#bdc3c7',
        mirror=True
    ),
    yaxis=dict(
        title='Value (USD)',
        gridcolor='#eef2f3',
        showline=True,
        linecolor='#bdc3c7',
        mirror=True,
        tickprefix='$'
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='#bdc3c7',
        borderwidth=1
    ),
    plot_bgcolor='white',
    hovermode='x unified',
    margin=dict(l=60, r=40, t=80, b=60),
    height=600
)

fig.show()

# Create the interactive figure for Volatility Difference
fig_diff = go.Figure()

# Add Volatility Difference trace
fig_diff.add_trace(go.Scatter(
    x=vega_Data.index,
    y=vega_Data['VRP ( Realized - VIX )'],
    name='VRP ( Realized - VIX )',
    line=dict(color='#8e44ad', width=2.0),
    hovertemplate='Vol Diff: %{y:.4f}<extra></extra>'
))

# Update layout for modern aesthetics and interactivity
fig_diff.update_layout(
    title=dict(
        text='VRP ( Realized - VIX )',
        font=dict(size=18, family='Arial', color='#2c3e50'),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title='Date',
        gridcolor='#eef2f3',
        showline=True,
        linecolor='#bdc3c7',
        mirror=True
    ),
    yaxis=dict(
        title='Difference (Volatility %)',
        gridcolor='#eef2f3',
        showline=True,
        linecolor='#bdc3c7',
        mirror=True
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='#bdc3c7',
        borderwidth=1
    ),
    plot_bgcolor='white',
    hovermode='x unified',
    margin=dict(l=60, r=40, t=80, b=60),
    height=400
)

fig_diff.show()